# AgroFarm AI: Data & Vector Store Setup

This notebook handles the initial data engineering pipeline for the AgroFarm AI project. It performs two primary functions:

1. **Field Data Simulation:** Merges static agronomy parameters with simulated live telemetry (moisture, irrigation status) to create a dynamic field dataset.
2. **Knowledge Base Vectorization:** Extracts text from raw agricultural PDFs and Word documents, chunks the text, computes TF-IDF embeddings, and builds a local ChromaDB vector store for Retrieval-Augmented Generation (RAG).

In [1]:
import logging
import pickle
from pathlib import Path
from typing import Dict, List, Tuple

import pandas as pd
import chromadb
from pypdf import PdfReader
from docx import Document
from sklearn.feature_extraction.text import TfidfVectorizer
from IPython.display import display
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Configure standard logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

# Define directories using modern pathlib
directories = [
    Path("../data/processed"),
    Path("../data/vectorstore")
]

# Safely initialize directories
try:
    for d in directories:
        d.mkdir(parents=True, exist_ok=True)
    logging.info("Environment imported and data directories safely verified.")
except PermissionError as e:
    logging.error(f"Permission denied while creating directories: {e}")
except Exception as e:
    logging.error(f"An unexpected error occurred during setup: {e}")

INFO: Environment imported and data directories safely verified.


## Part 1: Field Data Generation
Converting the static `base_agronomy_parameters.csv` into a dynamic `field_status.csv` simulating real-time farm conditions.

In [2]:
def generate_field_data(base_csv_path: Path, output_csv_path: Path) -> None:
    """Generates dynamic live-field data by merging static agronomy metrics with simulated field conditions."""
    
    FIELD_DEFINITIONS = [
        {"crop": "wheat",  "field_name": "Field 1 - North Plot",  "soil_moisture_pct": 22, "last_irrigated_days_ago": 5, "growth_stage": "Tillering",  "lat": 31.42, "lon": 73.09},
        {"crop": "wheat",  "field_name": "Field 2 - East Plot",   "soil_moisture_pct": 35, "last_irrigated_days_ago": 2, "growth_stage": "Heading",    "lat": 31.44, "lon": 73.10},
        {"crop": "cotton", "field_name": "Field 3 - South Plot",  "soil_moisture_pct": 40, "last_irrigated_days_ago": 3, "growth_stage": "Flowering",  "lat": 31.40, "lon": 73.08},
        {"crop": "cotton", "field_name": "Field 4 - West Plot",   "soil_moisture_pct": 18, "last_irrigated_days_ago": 7, "growth_stage": "Boll formation", "lat": 31.41, "lon": 73.12},
        {"crop": "rice",   "field_name": "Field 5 - River Plot",  "soil_moisture_pct": 55, "last_irrigated_days_ago": 1, "growth_stage": "Vegetative", "lat": 31.45, "lon": 73.07},
    ]

    try:
        if not base_csv_path.exists():
            raise FileNotFoundError(f"Source data not found at {base_csv_path}")
            
        df_raw = pd.read_csv(base_csv_path)
        output_rows = []
        crop_counters = {}

        for i, field_def in enumerate(FIELD_DEFINITIONS, start=1):
            crop = field_def["crop"]
            idx = crop_counters.get(crop, 0)
            crop_counters[crop] = idx + 1
            
            crop_data = df_raw[df_raw['label'] == crop]
            if crop_data.empty:
                logging.warning(f"No base data found for crop: {crop}. Skipping.")
                continue
                
            base_row = crop_data.iloc[idx % len(crop_data)].to_dict()
            
            field_status = {
                "field_id": f"F{i}",
                "field_name": field_def["field_name"],
                "crop": crop,
                "N": base_row["N"],
                "P": base_row["P"],
                "K": base_row["K"],
                "ph": round(base_row["ph"], 2),
                "soil_moisture_pct": field_def["soil_moisture_pct"],
                "last_irrigated_days_ago": field_def["last_irrigated_days_ago"],
                "growth_stage": field_def["growth_stage"],
                "latitude": field_def["lat"],
                "longitude": field_def["lon"],
            }
            output_rows.append(field_status)

        df_processed = pd.DataFrame(output_rows)
        df_processed.to_csv(output_csv_path, index=False)

        logging.info(f"Successfully built {output_csv_path.name} with {len(df_processed)} demo fields.")
        display(df_processed.head())

    except Exception as e:
        logging.error(f"Failed to generate field data: {e}")

In [3]:
# Execute Part 1
base_path = Path("../data/raw/base_agronomy_parameters.csv")
output_path = Path("../data/processed/field_status.csv")
generate_field_data(base_path, output_path)

INFO: Successfully built field_status.csv with 5 demo fields.


,field_id,field_name,crop,N,P,K,ph,soil_moisture_pct,last_irrigated_days_ago,growth_stage,latitude,longitude
0,F1,Field 1 - North Plot,wheat,95,55,45,6.50,22,5,Tillering,31.42,73.09
1,F2,Field 2 - East Plot,wheat,100,50,40,6.80,35,2,Heading,31.44,73.10
2,F3,Field 3 - South Plot,cotton,133,47,24,7.23,40,3,Flowering,31.40,73.08
3,F4,Field 4 - West Plot,cotton,136,36,20,6.93,18,7,Boll formation,31.41,73.12
4,F5,Field 5 - River Plot,rice,90,42,43,6.50,55,1,Vegetative,31.45,73.07


## Part 2: RAG Vector Store Compilation
Extracting domain knowledge from local agronomy documents, vectorizing via TF-IDF, and storing in a persistent ChromaDB instance.

In [4]:
# Global Hyperparameters & Metadata Mapping
CHUNK_SIZE: int = 800
CHUNK_OVERLAP: int = 100
COLLECTION_NAME: str = "agronomy_knowledge"

CROP_MAP: Dict[str, str] = {
    "wheat_growth_stages.docx": "wheat",
    "wheat_irrigation_problems.docx": "wheat",
    "cotton_irrigation_guide.pdf": "cotton",
    "rice_production_manual.pdf": "rice",
    "tomato_ipm_guide.pdf": "tomato",
    "integrated_pest_management_manual.pdf": "general_ipm",
}

def extract_text_from_pdf(filepath: Path) -> str:
    """Extract text from a PDF file page by page."""
    reader = PdfReader(str(filepath))
    extracted = [page.extract_text() or "" for page in reader.pages]
    return "\n".join(extracted).strip()

def extract_text_from_docx(filepath: Path) -> str:
    """Extract text from a Microsoft Word document."""
    doc = Document(str(filepath))
    extracted = [p.text for p in doc.paragraphs if p.text.strip()]
    return "\n".join(extracted).strip()

def chunk_document_text(text: str, chunk_size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP) -> List[str]:
    """
    Industry-standard semantic chunking. 
    Splits text dynamically by paragraphs, then sentences, keeping agronomy concepts intact.
    """
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=overlap,
        separators=["\n\n", "\n", ". ", " ", ""],
        length_function=len
    )
    return text_splitter.split_text(text)

In [5]:
def load_and_chunk_documents(source_dir: Path) -> Tuple[List[str], List[Dict[str, str]], List[str]]:
    """Traverse directory, extract text across documents, and generate chunk metadata."""
    if not source_dir.exists():
        source_dir.mkdir(parents=True, exist_ok=True)
        raise FileNotFoundError(f"Source directory created at {source_dir}. Place raw files there and rerun.")

    chunks, metadatas, ids = [], [], []
    doc_counter = 0
    supported_extensions = {".pdf", ".docx"}
    document_files = sorted([f for f in source_dir.iterdir() if f.suffix.lower() in supported_extensions])

    if not document_files:
        logging.warning(f"No PDF or DOCX files found in {source_dir}.")
        return chunks, metadatas, ids

    for file_path in document_files:
        crop_label = CROP_MAP.get(file_path.name, "general_ipm")
        try:
            raw_text = extract_text_from_pdf(file_path) if file_path.suffix.lower() == ".pdf" else extract_text_from_docx(file_path)
            doc_chunks = chunk_document_text(raw_text)
            logging.info(f"Parsed {file_path.name} -> {len(doc_chunks)} chunks [Crop: {crop_label}]")

            for chunk in doc_chunks:
                chunks.append(chunk)
                metadatas.append({"source": file_path.name, "crop": crop_label})
                ids.append(f"doc_{doc_counter}")
                doc_counter += 1

        except Exception as e:
            logging.error(f"Failed to process {file_path.name}: {e}")

    return chunks, metadatas, ids

def build_vectorstore(source_dir: Path, vectorstore_dir: Path, vectorizer_path: Path, batch_size: int = 200) -> None:
    """Fit TF-IDF embeddings and persist them into a local ChromaDB collection."""
    chunks, metadatas, ids = load_and_chunk_documents(source_dir)
    if not chunks:
        logging.error("Vectorization aborted: No text chunks extracted.")
        return

    logging.info(f"Fitting TF-IDF vectorizer on {len(chunks)} text chunks...")
    vectorizer = TfidfVectorizer(max_features=2000, stop_words="english")
    embeddings = vectorizer.fit_transform(chunks).toarray().tolist()

    with open(vectorizer_path, "wb") as f:
        pickle.dump(vectorizer, f)
    
    client = chromadb.PersistentClient(path=str(vectorstore_dir))
    try:
        client.delete_collection(COLLECTION_NAME)
    except Exception:
        pass

    collection = client.create_collection(name=COLLECTION_NAME, metadata={"hnsw:space": "cosine"})

    logging.info(f"Ingesting embeddings into ChromaDB collection '{COLLECTION_NAME}'...")
    for i in range(0, len(chunks), batch_size):
        collection.add(
            documents=chunks[i : i + batch_size],
            embeddings=embeddings[i : i + batch_size],
            metadatas=metadatas[i : i + batch_size],
            ids=ids[i : i + batch_size],
        )

    logging.info(f"Vector store successfully built and persisted at: {vectorstore_dir}")

    # Automated verification test query
    query_vector = vectorizer.transform(["wheat tillering irrigation schedule"]).toarray().tolist()
    sample_result = collection.query(query_embeddings=query_vector, n_results=1)
    if sample_result["documents"] and sample_result["documents"][0]:
        logging.info("Verification query passed: Retrieval engine is functioning as expected.")

In [6]:
# Execute Part 2
SOURCE_PATH = Path("../data/raw/agri_docs")
VECTORSTORE_PATH = Path("../data/vectorstore/agri_vectorstore")
VECTORIZER_PATH = Path("../data/vectorstore/tfidf_vectorizer.pkl")

build_vectorstore(SOURCE_PATH, VECTORSTORE_PATH, VECTORIZER_PATH)

INFO: Parsed cotton_irrigation_guide.pdf -> 11 chunks [Crop: cotton]
INFO: Parsed integrated_pest_management_manual.pdf -> 96 chunks [Crop: general_ipm]
INFO: Parsed rice_production_manual.pdf -> 112 chunks [Crop: rice]
INFO: Parsed tomato_ipm_guide.pdf -> 814 chunks [Crop: tomato]
INFO: Parsed wheat_growth_stages.docx -> 7 chunks [Crop: wheat]
INFO: Parsed wheat_irrigation_problems.docx -> 89 chunks [Crop: wheat]
INFO: Fitting TF-IDF vectorizer on 1129 text chunks...
INFO: Ingesting embeddings into ChromaDB collection 'agronomy_knowledge'...
INFO: Vector store successfully built and persisted at: ..\data\vectorstore\agri_vectorstore
INFO: Verification query passed: Retrieval engine is functioning as expected.
